In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS investment_pyspark.gold;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType


In [0]:
df_silver = spark.read.table("investment_pyspark.silver.holdings_clean")
df_silver.show()

In [0]:
df_gold_summary = (df_silver.select(
    F.sum("Invested_value").alias("total_invested"),
    F.sum("Current_value").alias("total_current_value"),
    F.sum("Unrealized_pnl").alias("total_unrealized_pnl"))
                   .withColumn(
                       "overall_return_pct",
                               F.round(
                                   F.col("total_unrealized_pnl")/F.col("total_invested")* 100,2).cast(DecimalType(18,2)))
                   .withColumn("gold_processed_timestamp", F.current_timestamp())
)

In [0]:
df_gold_summary.show()

In [0]:
df_gold_summary.write.format("delta").mode("overwrite").saveAsTable("investment_pyspark.gold.portfolio_summary")

In [0]:
%sql
select * from investment_pyspark.gold.portfolio_summary;